# Session 1 — Tokenization & Model Architecture Foundations**Duration:** 75 min  **Repository:** [MiniMind-Colab](https://github.com/your-org/minimind-colab)| Part | Topic | Time ||------|-------|------|| A | BPE Tokenizer (`train_tokenizer.py`) | 25 min || B | Model Architecture — RMSNorm, RoPE, GQA Attention (`model_minimind.py`) | 45 min || — | Wrap-up & Q&A | 5 min |

In [ ]:
# Cell 2 — Environment setup!pip install tokenizers torch --quietimport torchimport torch.nn as nnimport torch.nn.functional as Fimport mathprint(f'PyTorch {torch.__version__}  CUDA available: {torch.cuda.is_available()}')

## Part A — BPE Tokenizer (25 min)Byte-Pair Encoding (BPE) iteratively merges the most frequent byte pairsinto new tokens until the desired vocabulary size is reached.MiniMind uses a **6 400-token** vocabulary — tiny compared to GPT-2's 50 257but sufficient for a teaching model.Key concepts:- `ByteLevel` pre-tokenizer maps raw bytes → printable characters- Special tokens (`<|im_start|>`, `<|im_end|>`, …) are reserved first- After training, the tokenizer can encode / decode any UTF-8 string

In [ ]:
# Cell 4 — Train a BPE tokenizer on a small corpusfrom tokenizers import decoders, models, pre_tokenizers, trainers, TokenizerVOCAB_SIZE = 6400SPECIAL_TOKENS_NUM = 36# Build a small demo corpuscorpus = [    'The quick brown fox jumps over the lazy dog.',    'MiniMind is a tiny language model for teaching.',    'Byte-Pair Encoding merges frequent byte pairs.',    'Attention is all you need.',    'Transformers revolutionized natural language processing.',] * 100  # repeat so BPE has enough frequency signaltokenizer = Tokenizer(models.BPE())tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)# Reserve special tokensspecial_tokens = ['<|endoftext|>', '<|im_start|>', '<|im_end|>']extra_special = [f'<|reserved_{i}|>' for i in range(3, SPECIAL_TOKENS_NUM)]tool_tokens = ['<tool_call>', '</tool_call>', '<tool_response>',               '</tool_response>', '<think>', '</think>']all_special = special_tokens + extra_special + tool_tokenstrainer = trainers.BpeTrainer(    vocab_size=VOCAB_SIZE,    initial_alphabet=pre_tokenizers.ByteLevel.alphabet(),    special_tokens=all_special,)tokenizer.train_from_iterator(corpus, trainer=trainer)tokenizer.decoder = decoders.ByteLevel()print(f'Vocab size: {tokenizer.get_vocab_size()}')

In [ ]:
# Cell 5 — Inspect vocabularyvocab = tokenizer.get_vocab()sorted_tokens = sorted(vocab.items(), key=lambda x: x[1])print('First 20 tokens (includes special tokens):')for tok, idx in sorted_tokens[:20]:    print(f'  {idx:>5d}  {tok!r}')print(f'\nTotal merges learned: {tokenizer.get_vocab_size() - 256 - len(all_special)}')

In [ ]:
# Cell 6 — Chat-template demodef apply_chat_template(messages, tokenizer):    """Apply MiniMind chat template and encode."""    text = ''    for msg in messages:        role = msg['role']        content = msg['content']        text += f'<|im_start|>{role}\n{content}<|im_end|>\n'    return textmessages = [    {'role': 'user', 'content': 'Hello!'},    {'role': 'assistant', 'content': 'Hi there!'},]formatted = apply_chat_template(messages, tokenizer)print('Formatted text:')print(formatted)encoded = tokenizer.encode(formatted)print(f'\nToken IDs ({len(encoded.ids)} tokens): {encoded.ids[:30]}...')decoded = tokenizer.decode(encoded.ids)print(f'\nDecoded back: {decoded!r}')

In [ ]:
# Cell 7 — ✅ Milestone 1: Tokenizer encode/decode round-triptest_string = 'MiniMind is a tiny language model!'encoded_ids = tokenizer.encode(test_string).idsdecoded_string = tokenizer.decode(encoded_ids)assert decoded_string == test_string, (    f'Round-trip failed!\n'    f'  Original: {test_string!r}\n'    f'  Decoded:  {decoded_string!r}')print(f'✅ Milestone 1 passed — lossless encode/decode')print(f'   "{test_string}" → {encoded_ids} → "{decoded_string}"')

## Part B — Model Architecture (45 min)We now build the core components of a transformer decoder, following`model/model_minimind.py`.**MiniMindConfig defaults:**| Parameter | Value ||-----------|-------|| `hidden_size` | 768 || `num_hidden_layers` | 8 || `num_attention_heads` | 8 || `num_key_value_heads` | 4 || `head_dim` | 96 || `vocab_size` | 6 400 || `intermediate_size` | 2 432 |Components we will implement:1. **RMSNorm** — simpler alternative to LayerNorm2. **RoPE** — Rotary Position Embeddings3. **Grouped-Query Attention (GQA)** — memory-efficient attention

### 1. RMSNorm — Root Mean Square Layer NormalizationUnlike LayerNorm, RMSNorm does **not** re-center the activations(no mean subtraction). It only re-scales:$$\text{RMSNorm}(x) = \frac{x}{\sqrt{\text{mean}(x^2) + \epsilon}} \cdot \gamma$$where $\gamma$ is a learnable per-dimension scale parameter.**Why RMSNorm?** — Faster than LayerNorm (no mean computation),and empirically works just as well for transformer training.

In [ ]:
# Cell 9 — RMSNorm implementationclass RMSNorm(torch.nn.Module):    def __init__(self, dim, eps=1e-5):        super().__init__()        self.eps = eps        self.weight = nn.Parameter(torch.ones(dim))    def norm(self, x):        # ============================================================        # TODO: Implement RMS normalization        #        # Normalize x by its root-mean-square value.        #        # Input:  x — tensor of shape (..., dim)        # Output: tensor of same shape, each vector divided by its RMS        #        # Mathematical intuition:        #   RMS(x) = sqrt(mean(x^2) + eps)        #   output = x / RMS(x)        #        # Hint: use torch.rsqrt, x.pow(2), and .mean(-1, keepdim=True)        # ============================================================        raise NotImplementedError("Fill in the TODO above")    def forward(self, x):        return (self.weight * self.norm(x.float())).type_as(x)print('RMSNorm class defined ✓')

In [ ]:
# Cell 10 — Test RMSNormrms = RMSNorm(dim=768)x = torch.randn(2, 32, 768)out = rms(x)assert out.shape == x.shape, f'Shape mismatch: {out.shape} vs {x.shape}'# After normalization, RMS of each vector should be ≈ 1rms_vals = out.float().pow(2).mean(-1).sqrt()print(f'Output shape: {out.shape}')print(f'RMS values (should be ≈ 1): mean={rms_vals.mean():.4f}, std={rms_vals.std():.4f}')print('✓ RMSNorm test passed')

### 2. Rotary Position Embeddings (RoPE)RoPE encodes position by **rotating** query and key vectors in 2D sub-spaces.For dimension pair $(d_{2i}, d_{2i+1})$ at position $m$:$$\begin{pmatrix} q'_{2i} \\ q'_{2i+1} \end{pmatrix}= \begin{pmatrix} \cos(m\theta_i) & -\sin(m\theta_i) \\\sin(m\theta_i) & \cos(m\theta_i) \end{pmatrix}\begin{pmatrix} q_{2i} \\ q_{2i+1} \end{pmatrix}$$where $\theta_i = \text{base}^{-2i/d}$ and base = 1 000 000.**Key insight:** The dot product $q' \cdot k'$ depends only onthe *relative* distance between positions, giving the modeltranslation-invariant position awareness.

In [ ]:
# Cell 11 — RoPE: precompute frequencies & apply rotationdef precompute_freqs_cis(dim, end=32 * 1024, rope_base=1e6):    """Precompute cosine and sine tables for RoPE."""    freqs = 1.0 / (rope_base ** (torch.arange(0, dim, 2)[:(dim // 2)].float() / dim))    # ============================================================    # TODO: Build cosine and sine frequency tables    #    # Given the per-dimension frequencies `freqs` (shape [dim//2]),    # compute the full cos/sin tables for all positions 0..end-1.    #    # Input:  freqs — shape (dim // 2,)    # Output: freqs_cos — shape (end, dim)    #         freqs_sin — shape (end, dim)    #    # Mathematical intuition:    #   1. Create position indices t = [0, 1, ..., end-1]    #   2. Outer product: angles[pos, i] = t[pos] * freqs[i]    #   3. Duplicate cos/sin along last dim: cat([cos, cos], dim=-1)    #    # Hint: torch.arange, torch.outer, torch.cos, torch.sin, torch.cat    # ============================================================    raise NotImplementedError("Fill in the TODO above")def apply_rotary_pos_emb(q, k, cos, sin, unsqueeze_dim=1):    """Apply rotary embeddings to query and key tensors."""    # ============================================================    # TODO: Apply rotary position embeddings    #    # Rotate q and k using the precomputed cos and sin tables.    #    # Input:  q — (batch, seq_len, n_heads, head_dim)    #         k — (batch, seq_len, n_kv_heads, head_dim)    #         cos, sin — (seq_len, head_dim)    # Output: q_embed, k_embed — same shapes as q, k    #    # Mathematical intuition:    #   rotate_half(x) = [-x[..., d//2:], x[..., :d//2]]    #   q_embed = q * cos + rotate_half(q) * sin    #    # Hint: define rotate_half first, then apply:    #   q_embed = (q * cos.unsqueeze(unsqueeze_dim)) +    #             (rotate_half(q) * sin.unsqueeze(unsqueeze_dim))    # ============================================================    raise NotImplementedError("Fill in the TODO above")def repeat_kv(x, n_rep):    """Repeat KV heads to match the number of query heads (for GQA)."""    bs, slen, num_kv_heads, head_dim = x.shape    if n_rep == 1:        return x    return (x[:, :, :, None, :]            .expand(bs, slen, num_kv_heads, n_rep, head_dim)            .reshape(bs, slen, num_kv_heads * n_rep, head_dim))print('RoPE functions defined ✓')

In [ ]:
# Cell 12 — Test RoPEHEAD_DIM = 96SEQ_LEN = 32freqs_cos, freqs_sin = precompute_freqs_cis(dim=HEAD_DIM, end=SEQ_LEN)assert freqs_cos.shape == (SEQ_LEN, HEAD_DIM), f'freqs_cos shape: {freqs_cos.shape}'assert freqs_sin.shape == (SEQ_LEN, HEAD_DIM), f'freqs_sin shape: {freqs_sin.shape}'# Test apply_rotary_pos_embq = torch.randn(2, SEQ_LEN, 8, HEAD_DIM)k = torch.randn(2, SEQ_LEN, 4, HEAD_DIM)q_rot, k_rot = apply_rotary_pos_emb(q, k, freqs_cos, freqs_sin)assert q_rot.shape == q.shape, f'q_rot shape: {q_rot.shape}'assert k_rot.shape == k.shape, f'k_rot shape: {k_rot.shape}'print(f'freqs_cos shape: {freqs_cos.shape}')print(f'freqs_sin shape: {freqs_sin.shape}')print(f'q_rot shape:     {q_rot.shape}')print(f'k_rot shape:     {k_rot.shape}')print('✅ Milestone 2 passed — RMSNorm & RoPE shapes correct')

### 3. Grouped-Query Attention (GQA)Standard multi-head attention uses the same number of Q, K, V heads.**GQA** reduces memory by sharing K/V heads across groups of Q heads:- 8 query heads, 4 KV heads → each KV head serves 2 query heads- KV cache size is halved with minimal quality lossThe `repeat_kv` helper replicates KV heads to match the query head countbefore computing attention scores.

In [ ]:
# Cell 13 — Grouped-Query Attentionfrom dataclasses import dataclass@dataclassclass MiniMindConfig:    hidden_size: int = 768    num_hidden_layers: int = 8    num_attention_heads: int = 8    num_key_value_heads: int = 4    head_dim: int = 96    vocab_size: int = 6400    intermediate_size: int = 2432class Attention(nn.Module):    def __init__(self, config: MiniMindConfig):        super().__init__()        self.num_key_value_heads = config.num_key_value_heads or config.num_attention_heads        self.n_local_heads = config.num_attention_heads        self.n_local_kv_heads = self.num_key_value_heads        self.n_rep = self.n_local_heads // self.n_local_kv_heads        self.head_dim = config.head_dim        self.q_proj = nn.Linear(config.hidden_size,                                config.num_attention_heads * self.head_dim, bias=False)        self.k_proj = nn.Linear(config.hidden_size,                                self.num_key_value_heads * self.head_dim, bias=False)        self.v_proj = nn.Linear(config.hidden_size,                                self.num_key_value_heads * self.head_dim, bias=False)        self.o_proj = nn.Linear(config.num_attention_heads * self.head_dim,                                config.hidden_size, bias=False)        self.q_norm = RMSNorm(self.head_dim)        self.k_norm = RMSNorm(self.head_dim)    def forward(self, x, position_embeddings):        bsz, seq_len, _ = x.shape        # ============================================================        # TODO: Implement GQA forward pass        #        # Compute grouped-query attention from input x.        #        # Input:  x — (batch, seq_len, hidden_size)        #         position_embeddings — (cos, sin) each (seq_len, head_dim)        # Output: tensor of shape (batch, seq_len, hidden_size)        #        # Steps:        #   1. Project x → xq, xk, xv using self.q/k/v_proj        #   2. Reshape into heads:        #      xq: (bsz, seq_len, n_local_heads, head_dim)        #      xk: (bsz, seq_len, n_local_kv_heads, head_dim)        #   3. Apply self.q_norm / self.k_norm        #   4. Apply RoPE via apply_rotary_pos_emb(xq, xk, cos, sin)        #   5. Transpose to (bsz, heads, seq_len, head_dim)        #   6. repeat_kv on xk, xv to match query head count        #   7. scores = (xq @ xk^T) / sqrt(head_dim)        #   8. Add causal mask (upper-triangular -inf)        #   9. softmax → attn_weights @ xv        #  10. Merge heads → self.o_proj        #        # Hint: use self.q_proj, .view(), .transpose(1,2),        #       repeat_kv(), math.sqrt(), F.softmax()        # ============================================================        raise NotImplementedError("Fill in the TODO above")print('Attention class defined ✓')

In [ ]:
# Cell 14 — ✅ Milestone 2 & 3: Attention output shapeconfig = MiniMindConfig()attn = Attention(config)BATCH, SEQ_LEN, HIDDEN = 2, 32, config.hidden_sizex = torch.randn(BATCH, SEQ_LEN, HIDDEN)# Precompute position embeddings for the sequencefreqs_cos, freqs_sin = precompute_freqs_cis(dim=config.head_dim, end=SEQ_LEN)pos_emb = (freqs_cos, freqs_sin)with torch.no_grad():    out = attn(x, pos_emb)expected_shape = (BATCH, SEQ_LEN, HIDDEN)assert out.shape == expected_shape, (    f'Attention output shape mismatch: {out.shape} vs {expected_shape}')print(f'✅ Milestone 3 passed — Attention output shape: {out.shape}')print(f'   Expected: ({BATCH}, {SEQ_LEN}, {HIDDEN})')print(f'   Config: {config.num_attention_heads} Q heads, '      f'{config.num_key_value_heads} KV heads, '      f'head_dim={config.head_dim}')

## Wrap-up (5 min)### What we built today| Component | Key idea ||-----------|----------|| **BPE Tokenizer** | Iterative byte-pair merging; 6 400 tokens || **RMSNorm** | Normalize by RMS only (no mean subtraction) || **RoPE** | Encode position via rotation matrices || **GQA Attention** | Share KV heads across Q head groups |### Next session- Feed-Forward Network (SwiGLU)- Full Transformer Block- Stacking layers into the complete MiniMind model### References- [RMSNorm paper](https://arxiv.org/abs/1910.07467)- [RoPE paper (RoFormer)](https://arxiv.org/abs/2104.09864)- [GQA paper](https://arxiv.org/abs/2305.13245)